# Anomaly Detection with a Variational Autoencoder (PyTorch)

**The idea in one sentence:** train a generative model on *only normal data* so it learns what normal looks like, then flag any new point that the model finds surprising (assigns low probability to).

A plain autoencoder does this with reconstruction error. A **VAE** does something stronger: it gives you an actual (bound on the) log-likelihood $\log p(x)$, so "anomaly score" has a real probabilistic meaning instead of just being "the reconstruction looked bad".

### Roadmap

| Section | What happens |
|---|---|
| 1 | Build a synthetic sensor dataset with 3 kinds of anomaly |
| 2 | The math: ELBO, reparameterization trick, closed-form KL |
| 3 | Define the VAE in PyTorch |
| 4 | Train it (on normal data only) |
| 5 | Three ways to score anomalies, including an importance-weighted estimate of $\log p(x)$ |
| 6 | Pick a threshold and evaluate honestly |
| 7 | Look inside the latent space |
| 8 | How to swap in your own data + the traps |

Everything runs on CPU in under two minutes. No downloads.

---

## 0. Setup

Only `torch`, `numpy`, `matplotlib`, `scikit-learn`. If you're missing any:

```
pip install torch numpy matplotlib scikit-learn
```

In [ ]:
import math
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from sklearn.metrics import (
    roc_auc_score, average_precision_score, roc_curve,
    precision_recall_curve, confusion_matrix, classification_report,
)

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

# Your laptop has no CUDA, so this will pick CPU. The code is device-agnostic.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch:", torch.__version__, "| device:", device)

## 1. The data

We simulate a machine with 10 sensors. Crucially, the sensors are **not independent** — they are all driven by 3 hidden factors:

- $z_1$ = load on the machine
- $z_2$ = ambient conditions
- $z_3$ = wear

So the data really lives on a curved 3-dimensional surface (a *manifold*) inside 10-dimensional space, plus noise. This is the situation a VAE is built for: the latent dimension of the model is going to try to recover exactly this kind of structure.

### Three flavours of anomaly

| Type | What it is | Would a per-sensor min/max rule catch it? |
|---|---|---|
| `drift` | temperature and pressure slowly shift out of range | Yes, eventually |
| `spike` | vibration suddenly multiplies | Yes |
| `decoupled` | every sensor is individually in its normal range, but the **combination** is impossible (correlations broken) | **No** |

That third type is the whole argument for density modelling. Any single reading looks fine; only a model of the *joint* distribution notices something is wrong.

In [ ]:
FEATURES = ["temp_in", "temp_out", "pressure", "flow_rate", "vib_x",
            "vib_y", "rpm", "current", "voltage", "humidity"]
D = len(FEATURES)

# Per-sensor measurement noise (different scales on purpose - realistic, and it
# forces us to standardise later).
NOISE = np.array([0.4, 0.5, 0.3, 1.0, 0.02, 0.02, 8.0, 0.15, 0.6, 0.8])


def _sensors_from_latents(z1, z2, z3):
    # z1 = load, z2 = ambient, z3 = wear. Deliberately NON-linear.
    temp_in  = 60.0 + 5.0 * z2 + 2.0 * np.tanh(z1)
    temp_out = temp_in + 12.0 + 6.0 * np.sin(1.5 * z1) + 0.5 * z3
    pressure = 30.0 + 8.0 * np.cos(z1) - 1.5 * z3          # non-monotonic in load
    flow     = 100.0 + 20.0 * np.tanh(1.5 * z1) - 3.0 * z3 + 2.0 * z1 * z2
    vib_x    = 0.50 + 0.30 * z1 ** 2 + 0.10 * z3           # quadratic
    vib_y    = 0.40 + 0.25 * z1 ** 2
    rpm      = 1500.0 + 120.0 * z1
    current  = 10.0 + 1.2 * z1 + 0.4 * z3 + 0.5 * z1 ** 2
    voltage  = 230.0 - 0.8 * z1 - 0.6 * z2 * z3            # interaction
    humidity = 45.0 + 10.0 * z2
    return np.column_stack([temp_in, temp_out, pressure, flow, vib_x,
                            vib_y, rpm, current, voltage, humidity])


def sample_normal(n, rng):
    z1, z2, z3 = rng.normal(size=(3, n))
    X = _sensors_from_latents(z1, z2, z3)
    return X + rng.normal(0.0, NOISE, size=X.shape)


def sample_anomalies(n, rng):
    kinds = rng.choice(["drift", "spike", "decoupled"], size=n)
    z1, z2, z3 = rng.normal(size=(3, n))
    X = _sensors_from_latents(z1, z2, z3)
    X = X + rng.normal(0.0, NOISE, size=X.shape)

    m = kinds == "drift"                      # slow shift out of range
    X[m, 1] += rng.uniform(6, 12, m.sum())    # temp_out up
    X[m, 2] -= rng.uniform(5, 9, m.sum())     # pressure down

    m = kinds == "spike"                      # bearing failure
    X[m, 4] *= rng.uniform(3, 6, m.sum())
    X[m, 5] *= rng.uniform(3, 6, m.sum())

    # Correlations destroyed: shuffle each column independently so every
    # marginal distribution stays EXACTLY normal, but the joint is nonsense.
    m = kinds == "decoupled"
    if m.sum() > 0:
        block = sample_normal(m.sum(), rng)
        for d in range(D):
            block[:, d] = block[rng.permutation(m.sum()), d]
        X[m] = block
    return X, kinds


rng = np.random.default_rng(SEED)

X_train      = sample_normal(6000, rng)                 # normal only -> the VAE sees this
X_val_norm   = sample_normal(1500, rng)                 # normal only -> used to pick a threshold
X_test_norm  = sample_normal(2000, rng)
X_test_anom, test_kinds = sample_anomalies(400, rng)    # 400 / 2400 = ~17% anomalies

X_test = np.vstack([X_test_norm, X_test_anom])
y_test = np.concatenate([np.zeros(len(X_test_norm)), np.ones(len(X_test_anom))])

print("train (normal only):", X_train.shape)
print("val   (normal only):", X_val_norm.shape)
print("test :", X_test.shape, "| anomaly rate:", y_test.mean().round(3))

### Standardise — and fit the scaler on training data only

Sensors have wildly different units (rpm ~1500, vibration ~0.5). Without scaling, the reconstruction loss would be dominated by `rpm` and the model would ignore vibration entirely.

Fit the scaler on **train only**. Using test statistics is a leak, and here it would be a particularly nasty one: the anomalies would shift the mean and variance you standardise by.

In [ ]:
mu_tr = X_train.mean(axis=0)
sd_tr = X_train.std(axis=0)

def scale(X):
    return (X - mu_tr) / sd_tr

Xtr = torch.tensor(scale(X_train),     dtype=torch.float32)
Xva = torch.tensor(scale(X_val_norm),  dtype=torch.float32)
Xte = torch.tensor(scale(X_test),      dtype=torch.float32)

print("after scaling -> mean ~0, std ~1:",
      round(Xtr.mean().item(), 4), round(Xtr.std().item(), 4))

In [ ]:
# A quick look. Left: the manifold is visible as a curved band.
# Right: 'decoupled' anomalies sit INSIDE the marginal range of both axes.
anom_mask = y_test == 1
fig, ax = plt.subplots(1, 2, figsize=(11, 4.2))

ax[0].scatter(X_train[:, 6], X_train[:, 2], s=4, alpha=0.3)
ax[0].set(xlabel="rpm", ylabel="pressure", title="Training data (normal only)")

ax[1].scatter(X_test[~anom_mask, 6], X_test[~anom_mask, 2], s=6, alpha=0.25, label="normal")
for k, c in zip(["drift", "spike", "decoupled"], ["tab:orange", "tab:green", "tab:red"]):
    m = anom_mask.copy()
    m[anom_mask] = test_kinds == k
    ax[1].scatter(X_test[m, 6], X_test[m, 2], s=14, c=c, label=k)
ax[1].set(xlabel="rpm", ylabel="pressure", title="Test data")
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. The math, briefly

**Autoencoder recap.** Encoder $x \mapsto z$, decoder $z \mapsto \hat{x}$, minimise $\|x - \hat{x}\|^2$. The bottleneck forces it to keep only what matters. But $z$ is just *some* vector — the latent space has no probabilistic meaning, and you cannot ask "how likely is this $x$?"

**VAE.** We posit a generative story:

$$z \sim \mathcal{N}(0, I), \qquad x \mid z \sim \mathcal{N}\big(\mu_\theta(z),\, \sigma_x^2 I\big)$$

We want $\log p_\theta(x) = \log \int p_\theta(x \mid z)\, p(z)\, dz$. That integral is intractable, so we introduce an **encoder** $q_\phi(z \mid x) = \mathcal{N}(\mu_\phi(x), \operatorname{diag}\sigma^2_\phi(x))$ and use Jensen's inequality:

$$\log p_\theta(x) \;\ge\; \underbrace{\mathbb{E}_{q_\phi(z\mid x)}\big[\log p_\theta(x \mid z)\big] \;-\; D_{\mathrm{KL}}\big(q_\phi(z \mid x)\,\|\,p(z)\big)}_{\text{ELBO}}$$

The gap between the two sides is exactly $D_{\mathrm{KL}}(q_\phi(z\mid x) \| p_\theta(z \mid x)) \ge 0$.

Read the two terms as: **reconstruct well**, but **keep the posterior close to the prior**. The second term is the regulariser that a plain autoencoder lacks.

**Closed-form KL.** For two diagonal Gaussians with $p = \mathcal{N}(0,I)$:

$$D_{\mathrm{KL}} = -\tfrac{1}{2} \sum_{j=1}^{L} \left(1 + \log \sigma_j^2 - \mu_j^2 - \sigma_j^2\right)$$

No sampling needed. We parameterise $\log \sigma^2$ (`logvar`) rather than $\sigma$ so the network output is unconstrained on $\mathbb{R}$.

**Reparameterization trick.** We need a gradient through a sampling step. Writing

$$z = \mu_\phi(x) + \sigma_\phi(x) \odot \varepsilon, \qquad \varepsilon \sim \mathcal{N}(0, I)$$

moves the randomness into $\varepsilon$, which carries no parameters, so $\nabla_\phi$ flows straight through $\mu$ and $\sigma$. Sampling `z ~ N(mu, sigma)` directly would break the graph.

**Reconstruction term.** With a Gaussian decoder,

$$-\log p_\theta(x \mid z) = \sum_{d=1}^{D}\left[\tfrac{1}{2}\log(2\pi) + \log \sigma_x + \frac{(x_d - \mu_d(z))^2}{2\sigma_x^2}\right]$$

If you fix $\sigma_x = 1$ this collapses to $\tfrac{1}{2}\|x - \hat x\|^2 + \text{const}$ — plain MSE. We'll *learn* $\sigma_x$ instead, because it makes the ELBO a genuine log-density bound (units of nats), which is what we want for scoring. For binary data like MNIST pixels you'd use a Bernoulli decoder and BCE instead.

## 3. The model

Small on purpose. `latent_dim=3` matches the true number of hidden factors — in real work you'd tune it, and Section 8 says how.

In [ ]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim=64, latent_dim=3):
        super().__init__()
        self.latent_dim = latent_dim

        # ---- encoder: x -> hidden ----
        self.enc = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
        )
        # two heads: the posterior mean and log-variance
        self.fc_mu     = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)

        # ---- decoder: z -> reconstructed x ----
        self.dec = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim), nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )
        # one global learnable observation noise, shared across features
        # (data is standardised, so a single scalar is reasonable).
        self.raw_log_sigma_x = nn.Parameter(torch.zeros(1))

    @property
    def log_sigma_x(self):
        return self.raw_log_sigma_x.clamp(-4.0, 4.0)   # clamp = numerical safety

    def encode(self, x):                 # x: (B, D)
        h = self.enc(x)                  # (B, H)
        return self.fc_mu(h), self.fc_logvar(h)        # both (B, L)

    @staticmethod
    def reparameterize(mu, logvar):
        std = torch.exp(0.5 * logvar)    # sigma = exp(logvar / 2)
        eps = torch.randn_like(std)      # the trick: randomness has no parameters
        return mu + eps * std

    def decode(self, z):                 # z: (B, L)
        return self.dec(z)               # (B, D)

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar, z


model = VAE(input_dim=D, hidden_dim=64, latent_dim=3).to(device)
print(model)
print("trainable parameters:", sum(p.numel() for p in model.parameters()))

In [ ]:
LOG2PI = math.log(2.0 * math.pi)


def gaussian_nll(x, x_mu, log_sigma):
    # -log p(x|z), summed over features -> shape (B,)
    return (0.5 * LOG2PI + log_sigma
            + 0.5 * ((x - x_mu) / log_sigma.exp()) ** 2).sum(dim=1)


def kl_to_standard_normal(mu, logvar):
    # KL( N(mu, sigma^2) || N(0, I) ), summed over latent dims -> shape (B,)
    return -0.5 * (1.0 + logvar - mu.pow(2) - logvar.exp()).sum(dim=1)


def vae_loss(x, x_mu, mu, logvar, log_sigma, beta=1.0):
    rec = gaussian_nll(x, x_mu, log_sigma)
    kl  = kl_to_standard_normal(mu, logvar)
    loss = (rec + beta * kl).mean()        # negative ELBO, averaged over the batch
    return loss, rec.mean().item(), kl.mean().item()

## 4. Training

Standard loop. Two things worth noticing:

- We track validation loss on **held-out normal data**, not on the test set. The test set contains anomalies and must stay untouched until the very end.
- `beta` scales the KL term ($\beta$-VAE). $\beta = 1$ is the true ELBO. Lower $\beta$ reconstructs better but regularises less; higher $\beta$ pushes toward a cleaner latent space and risks *posterior collapse* (see Section 8).

In [ ]:
EPOCHS = 120
BATCH  = 256
BETA   = 1.0

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
Xtr_d, Xva_d = Xtr.to(device), Xva.to(device)
n = len(Xtr_d)

history = {"train": [], "val": [], "rec": [], "kl": [], "sigma_x": []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(n, device=device)
    tot = rec_tot = kl_tot = 0.0

    for i in range(0, n, BATCH):
        xb = Xtr_d[perm[i:i + BATCH]]

        x_mu, mu, logvar, _ = model(xb)
        loss, rec, kl = vae_loss(xb, x_mu, mu, logvar, model.log_sigma_x, BETA)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)   # cheap stability
        opt.step()

        w = len(xb) / n
        tot += loss.item() * w; rec_tot += rec * w; kl_tot += kl * w

    model.eval()
    with torch.no_grad():
        x_mu, mu, logvar, _ = model(Xva_d)
        val, _, _ = vae_loss(Xva_d, x_mu, mu, logvar, model.log_sigma_x, BETA)

    history["train"].append(tot);       history["val"].append(val.item())
    history["rec"].append(rec_tot);     history["kl"].append(kl_tot)
    history["sigma_x"].append(model.log_sigma_x.exp().item())

    if epoch % 20 == 0 or epoch == 1:
        print(f"epoch {epoch:4d} | train {tot:8.3f} | val {val.item():8.3f} "
              f"| rec {rec_tot:8.3f} | KL {kl_tot:6.3f} | sigma_x {history['sigma_x'][-1]:.3f}")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))

ax[0].plot(history["train"], label="train")
ax[0].plot(history["val"], label="val (normal)")
ax[0].set(xlabel="epoch", ylabel="negative ELBO (nats)", title="Loss")
ax[0].legend()

ax[1].plot(history["rec"], label="reconstruction", color="tab:orange")
ax[1].plot(history["kl"],  label="KL",             color="tab:green")
ax[1].set(xlabel="epoch", title="The two terms"); ax[1].legend()

ax[2].plot(history["sigma_x"], color="tab:purple")
ax[2].set(xlabel="epoch", ylabel=r"$\sigma_x$", title="Learned observation noise")

plt.tight_layout(); plt.show()

**How to read this.** The KL term should settle at a positive value clearly above zero — that means the latent code is actually carrying information about $x$. If KL drifts to ~0, the decoder has learned to ignore $z$ entirely (posterior collapse) and your anomaly scores will be worthless.

$\sigma_x$ shrinking tells you the model is fitting the normal data tightly. It also sets the scale of the reconstruction term, which is why learning it beats guessing it.

## 5. Scoring: three options

All three give **high = more anomalous**.

**(a) Reconstruction error.** Encode to $\mu_\phi(x)$ (no sampling), decode, measure squared error. Fast, intuitive, and what most tutorials stop at. It throws away the KL term, so it misses points that reconstruct fine but sit in an implausible region of latent space.

**(b) Negative ELBO.** The full training objective evaluated per sample. Uses both terms. Cheap.

**(c) Importance-weighted negative log-likelihood.** The principled one. Draw $K$ samples $z_k \sim q_\phi(z \mid x)$ and form

$$\log p(x) \;\approx\; \log \frac{1}{K}\sum_{k=1}^{K} \frac{p_\theta(x \mid z_k)\, p(z_k)}{q_\phi(z_k \mid x)}$$

This is a tighter lower bound than the ELBO and gets tighter as $K$ grows (it *is* the ELBO at $K=1$). Score $= -\log p(x)$, in nats. Costs $K$ decoder passes, which is nothing at inference time for tabular data.

Use (c) when you can afford it. It answers *how surprising is this point*, rather than *how badly did we redraw it*.

In [ ]:
@torch.no_grad()
def score_reconstruction(model, X, batch=1024):
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].to(device)
        mu, _ = model.encode(xb)
        x_hat = model.decode(mu)                    # deterministic: use the mean
        out.append(((xb - x_hat) ** 2).sum(dim=1).cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def score_neg_elbo(model, X, n_samples=8, batch=1024):
    model.eval()
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].to(device)
        mu, logvar = model.encode(xb)
        kl = kl_to_standard_normal(mu, logvar)
        rec = 0.0
        for _ in range(n_samples):                  # average the MC estimate
            z = model.reparameterize(mu, logvar)
            rec = rec + gaussian_nll(xb, model.decode(z), model.log_sigma_x)
        out.append((rec / n_samples + kl).cpu())
    return torch.cat(out).numpy()


@torch.no_grad()
def score_iw_nll(model, X, K=128, batch=256):
    # -log p(x), importance-weighted with K samples from q(z|x)
    model.eval()
    log_sigma = model.log_sigma_x
    out = []
    for i in range(0, len(X), batch):
        xb = X[i:i + batch].to(device)
        B = xb.shape[0]
        mu, logvar = model.encode(xb)                       # (B, L)
        std = torch.exp(0.5 * logvar)

        eps = torch.randn(K, B, model.latent_dim, device=device)
        z   = mu.unsqueeze(0) + eps * std.unsqueeze(0)      # (K, B, L)

        x_mu = model.decode(z.reshape(K * B, -1)).reshape(K, B, -1)   # (K, B, D)

        log_p_x_z = -(0.5 * LOG2PI + log_sigma
                      + 0.5 * ((xb.unsqueeze(0) - x_mu) / log_sigma.exp()) ** 2).sum(-1)
        log_p_z   = (-0.5 * LOG2PI - 0.5 * z ** 2).sum(-1)
        log_q_z_x = (-0.5 * LOG2PI - 0.5 * logvar.unsqueeze(0) - 0.5 * eps ** 2).sum(-1)

        log_w = log_p_x_z + log_p_z - log_q_z_x             # (K, B)
        log_px = torch.logsumexp(log_w, dim=0) - math.log(K)
        out.append((-log_px).cpu())
    return torch.cat(out).numpy()


scorers = {
    "reconstruction error": score_reconstruction,
    "negative ELBO":        score_neg_elbo,
    "IW neg log-lik":       score_iw_nll,
}

val_scores  = {k: f(model, Xva) for k, f in scorers.items()}
test_scores = {k: f(model, Xte) for k, f in scorers.items()}

for k in scorers:
    print(f"{k:22s} AUROC {roc_auc_score(y_test, test_scores[k]):.4f}"
          f"  AUPRC {average_precision_score(y_test, test_scores[k]):.4f}")

## 6. Picking a threshold, and evaluating

AUROC is threshold-free, which is nice for comparing scorers but useless in production — you have to actually decide where to cut.

The clean way: take the $q$-th percentile of scores on **held-out normal data**. Set $q = 99$ and you've chosen a ~1% false-alarm rate *by construction*, without ever looking at an anomaly. That matters, because in real deployments you usually don't have labelled anomalies to tune on.

In [ ]:
SCORER = "IW neg log-lik"
Q = 99.0

threshold = np.percentile(val_scores[SCORER], Q)
s = test_scores[SCORER]
y_pred = (s > threshold).astype(int)

print(f"scorer: {SCORER}")
print(f"threshold = {Q}th percentile of validation-normal scores = {threshold:.2f} nats\n")
print(classification_report(y_test, y_pred, target_names=["normal", "anomaly"], digits=3))

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
print(f"true neg {tn:5d}   false pos {fp:4d}")
print(f"false neg {fn:4d}   true pos  {tp:4d}")
print(f"\nrealised false-alarm rate: {fp / (fp + tn):.3%}  (we targeted {100 - Q:.1f}%)")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

# --- score distributions ---
lo, hi = np.percentile(s, 0.5), np.percentile(s, 99.5)
bins = np.linspace(lo, hi, 60)
ax[0].hist(s[y_test == 0], bins=bins, alpha=0.6, label="normal", density=True)
ax[0].hist(s[y_test == 1], bins=bins, alpha=0.6, label="anomaly", density=True)
ax[0].axvline(threshold, color="k", ls="--", label=f"threshold (p{Q:.0f})")
ax[0].set(xlabel="anomaly score (nats)", title="Score distributions"); ax[0].legend(fontsize=8)

# --- ROC, all three scorers ---
for k in scorers:
    fpr, tpr, _ = roc_curve(y_test, test_scores[k])
    ax[1].plot(fpr, tpr, label=f"{k} ({roc_auc_score(y_test, test_scores[k]):.3f})")
ax[1].plot([0, 1], [0, 1], "k:", lw=1)
ax[1].set(xlabel="false positive rate", ylabel="true positive rate", title="ROC")
ax[1].legend(fontsize=8)

# --- precision / recall ---
for k in scorers:
    prec, recl, _ = precision_recall_curve(y_test, test_scores[k])
    ax[2].plot(recl, prec, label=f"{k} ({average_precision_score(y_test, test_scores[k]):.3f})")
ax[2].axhline(y_test.mean(), color="k", ls=":", lw=1, label="chance")
ax[2].set(xlabel="recall", ylabel="precision", title="Precision-Recall")
ax[2].legend(fontsize=8)

plt.tight_layout(); plt.show()

In [ ]:
# Which anomaly types do we actually catch? This is the interesting table.
print(f"{'type':<12}{'n':>6}{'recall':>10}{'median score':>15}")
print("-" * 43)
for k in ["drift", "spike", "decoupled"]:
    idx = np.where(y_test == 1)[0][test_kinds == k]
    print(f"{k:<12}{len(idx):>6}{y_pred[idx].mean():>10.3f}{np.median(s[idx]):>15.1f}")
idx = np.where(y_test == 0)[0]
print(f"{'normal':<12}{len(idx):>6}{y_pred[idx].mean():>10.3f}{np.median(s[idx]):>15.1f}")

Look at the `decoupled` row. Those points have every sensor sitting comfortably inside its normal range — a per-sensor alarm system would see nothing at all — yet the VAE scores them as wildly improbable, because it has learned that `rpm` high *and* `pressure` low simply don't co-occur.

That is the payoff of modelling the joint density instead of ten marginals.

## 7. Inside the latent space

Two useful checks:

1. Normal data should map roughly onto the prior $\mathcal{N}(0, I)$ — a blob centred at the origin. That's the KL term doing its job.
2. Anomalies often get pushed to the edges, but not always. `spike`-type anomalies can land in a perfectly ordinary latent position and still reconstruct badly. That's *why* the score combines both terms rather than only measuring latent distance.

In [ ]:
with torch.no_grad():
    mu_te, _ = model.encode(Xte.to(device))
mu_te = mu_te.cpu().numpy()

fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))

ax[0].scatter(mu_te[y_test == 0, 0], mu_te[y_test == 0, 1], s=6, alpha=0.25, label="normal")
for k, c in zip(["drift", "spike", "decoupled"], ["tab:orange", "tab:green", "tab:red"]):
    m = np.zeros(len(y_test), bool); m[y_test == 1] = test_kinds == k
    ax[0].scatter(mu_te[m, 0], mu_te[m, 1], s=14, c=c, label=k)
ax[0].set(xlabel="$z_1$", ylabel="$z_2$", title=r"Posterior means $\mu_\phi(x)$")
ax[0].legend(fontsize=8)

sc = ax[1].scatter(mu_te[:, 0], mu_te[:, 1], c=np.log(s - s.min() + 1), s=8, cmap="viridis")
ax[1].set(xlabel="$z_1$", ylabel="$z_2$", title="Coloured by anomaly score")
plt.colorbar(sc, ax=ax[1], label="log score")
plt.tight_layout(); plt.show()

In [ ]:
# Per-feature reconstruction error tells you WHICH sensor is misbehaving -
# a cheap and very useful explanation to hand an operator.
with torch.no_grad():
    xb = Xte.to(device)
    mu_, _ = model.encode(xb)
    err = ((xb - model.decode(mu_)) ** 2).cpu().numpy()

fig, ax = plt.subplots(figsize=(10, 4))
w = 0.2
for i, (k, c) in enumerate(zip(["drift", "spike", "decoupled"],
                               ["tab:orange", "tab:green", "tab:red"])):
    m = np.zeros(len(y_test), bool); m[y_test == 1] = test_kinds == k
    ax.bar(np.arange(D) + (i - 1) * w, err[m].mean(0), width=w, label=k, color=c)
ax.bar(np.arange(D) + 2 * w, err[y_test == 0].mean(0), width=w, label="normal", color="tab:blue")
ax.set_xticks(range(D)); ax.set_xticklabels(FEATURES, rotation=45, ha="right")
ax.set(ylabel="mean squared error", title="Which sensor is responsible?")
ax.set_yscale("log"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. Using this on your own data

### The swap

Replace the data section with your own array and keep everything else:

```python
X = your_dataframe[feature_cols].to_numpy(dtype=np.float32)

# Train on data you believe is clean. If you can't be sure it's clean,
# a small contamination fraction (<5%) is usually survivable.
X_train, X_val_norm = train_test_split(X_clean, test_size=0.2, random_state=0)
```

Then set `D = X.shape[1]` and rebuild the model. For images, swap the `Linear` stacks for `Conv2d`/`ConvTranspose2d` and use a Bernoulli decoder (`F.binary_cross_entropy(..., reduction="none").sum(1)`) instead of the Gaussian NLL. For sequences, feed sliding windows or use an LSTM encoder.

### Hyperparameters that actually matter

| Knob | Effect | Where to start |
|---|---|---|
| `latent_dim` | Too large → the model memorises anomalies too and stops flagging them. Too small → underfits, everything looks anomalous. | 2–16 for tabular; tune by validation ELBO |
| `beta` | ↑ smoother latent space, ↓ reconstruction sharpness | 1.0, then try 0.5 / 2.0 |
| `hidden_dim`, depth | Capacity | Keep it small — a VAE that fits *too* well generalises to anomalies |
| `K` in the IW score | Tightness of the log-likelihood bound | 64–256 |
| threshold percentile | Directly sets your false-alarm rate | Pick from the cost of a false alarm vs a miss |

### Traps

**Posterior collapse.** KL drops to ~0, the decoder ignores $z$, and scores become meaningless. Watch the KL curve. Fixes: KL annealing (ramp `beta` from 0 to 1 over the first ~20 epochs), a smaller decoder, or free bits.

**The VAE reconstructs anomalies too well.** A known failure: high-capacity generative models sometimes assign *higher* likelihood to out-of-distribution data than to in-distribution data. Shrink the model, and check per-type recall like we did above rather than trusting one aggregate AUROC.

**Contaminated training data.** If anomalies leak into training, the model learns them as normal. Iterative cleaning helps: train, drop the top ~1% highest-scoring training points, retrain.

**Scaling and leakage.** Fit the scaler on train only, always. Heavy-tailed features often need a log or rank-Gaussian transform first, or the reconstruction term chases outliers.

**Non-stationarity.** Real sensor "normal" drifts with seasons and equipment age. Plan to retrain on a rolling window, and monitor the false-alarm rate against the percentile you targeted.

### Is a VAE even the right tool?

Be honest about baselines. For tabular anomaly detection, **Isolation Forest** and a plain **PCA reconstruction error** are often competitive and take one line each. The VAE earns its keep when the normal manifold is genuinely non-linear, when you want calibrated likelihoods, or when you also want to *generate* normal samples. Run the cheap baselines first — if PCA matches your VAE, the extra machinery isn't paying for itself.

```python
from sklearn.ensemble import IsolationForest
iso = IsolationForest(random_state=0).fit(scale(X_train))
print(roc_auc_score(y_test, -iso.score_samples(scale(X_test))))
```

In [ ]:
# The baseline check, actually run. Never skip this step.
from sklearn.ensemble import IsolationForest
from sklearn.decomposition import PCA

iso = IsolationForest(n_estimators=200, random_state=SEED).fit(scale(X_train))
iso_score = -iso.score_samples(scale(X_test))

pca = PCA(n_components=3).fit(scale(X_train))
Z = scale(X_test)
pca_score = ((Z - pca.inverse_transform(pca.transform(Z))) ** 2).sum(axis=1)

print(f"{'method':<24}{'AUROC':>8}{'AUPRC':>8}")
print("-" * 40)
for name, sc_ in [("PCA (3 comp) recon", pca_score),
                  ("Isolation Forest", iso_score),
                  *[(f"VAE: {k}", test_scores[k]) for k in scorers]]:
    print(f"{name:<24}{roc_auc_score(y_test, sc_):>8.4f}"
          f"{average_precision_score(y_test, sc_):>8.4f}")

---

## Summary

1. Train a VAE on **normal data only**; it learns $p(x)$ for normal behaviour.
2. The ELBO gives you a lower bound on $\log p(x)$; the importance-weighted estimate gives you a tighter one.
3. Score $= -\log p(x)$. High score = surprising = anomalous.
4. Set the threshold from a percentile of held-out **normal** scores, so you control the false-alarm rate directly.
5. Always check per-anomaly-type recall and compare against PCA / Isolation Forest before declaring victory.

**Good next experiments:** set `latent_dim=10` and watch recall degrade as the model gains enough capacity to reconstruct anomalies; set `BETA=0` and see the latent space stop resembling the prior; add 3% anomalies to `X_train` and measure the damage.